# Enterprise Multi-Agent AI System — Blackwell 6000

## Architecture: 1 CEO + 24 Department Specialists

### Flow
```
Objective → CEO Core Reasoning Model
  ├── Legal Compliance         ├── Accounting Audit
  ├── Finance Funding          ├── Design Production
  ├── Logistics Warehouse      ├── Sales Customer Service
  ├── Marketing                ├── IT
  ├── Operation Process         ├── Tax
  ├── Risk Management          ├── Strategic Planning
  ├── Procurement              ├── Investment
  ├── Quality Control          ├── Corporate Communications
  ├── Fixed Asset Management    ├── Internal Audit
  ├── Supply Chain Planning    ├── Data Analytics
  ├── Treasury Management      ├── Business Intelligence
  ├── Contract Administration   └── Asset Valuation
  → CEO aggregates, reviews, final decision
```

### Features
- Checkpoint every 100 steps → auto-upload to HF
- Resume from HF on session failure
- Error-free verified streaming datasets
- Blackwell BF16 + FSDP optimized

In [ ]:
import os, json, shutil, glob, math, random, sys
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, IterableDataset

from transformers import LlamaConfig, LlamaModel
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
from huggingface_hub import HfApi, hf_hub_download, login
from accelerate import Accelerator
import getpass

print("Imports OK")

In [ ]:
token = getpass.getpass("HF token: ")
login(token, add_to_git_credential=False)
api = HfApi()
print("Logged in!")

---
## 1. Config

In [ ]:
@dataclass
class Department:
    id: int
    name: str
    short: str
    description: str
    data_sources: List[str]

DEPARTMENTS = [
    Department(1,  "Legal Compliance",        "legal",     "Regulatory compliance, contract law", ["nvidia/Nemotron-Pretraining-Legal-v1"]),
    Department(2,  "Accounting Audit",        "audit",     "Financial audit, internal controls, GAAP", ["infCapital/investopedia_terms_en"]),
    Department(3,  "Finance Funding",         "finance",   "Capital structure, fundraising", ["JanosAudran/financial-reports-sec"]),
    Department(4,  "Design Production",       "design",    "Product design, manufacturing", ["HuggingFaceFW/fineweb-edu"]),
    Department(5,  "Logistics Warehouse",     "logistics", "Supply chain, inventory, shipping", ["HuggingFaceFW/fineweb-edu"]),
    Department(6,  "Sales Customer Service",  "sales",     "CRM, sales pipeline, customer support", ["HuggingFaceFW/fineweb-edu"]),
    Department(7,  "Marketing",               "marketing", "Brand strategy, campaigns, market research", ["HuggingFaceFW/fineweb-edu"]),
    Department(8,  "IT",                     "it",        "Infrastructure, cybersecurity", ["transformersbook/codeparrot"]),
    Department(9,  "Operation Process",       "ops",       "Business processes, workflow optimization", ["HuggingFaceFW/fineweb-edu"]),
    Department(10, "Tax",                    "tax",       "Tax planning, compliance, filing", ["infCapital/investopedia_terms_en"]),
    Department(11, "Risk Management",         "risk",      "Risk assessment, mitigation, insurance", ["HuggingFaceFW/fineweb-edu"]),
    Department(12, "Strategic Planning",      "strategy",  "Corporate strategy, M&A, expansion", ["HuggingFaceFW/fineweb-edu"]),
    Department(13, "Procurement",             "procure",   "Vendor management, sourcing", ["HuggingFaceFW/fineweb-edu"]),
    Department(14, "Investment",              "invest",    "Portfolio management, due diligence", ["JanosAudran/financial-reports-sec"]),
    Department(15, "Quality Control",         "quality",   "Quality assurance, testing, compliance", ["HuggingFaceFW/fineweb-edu"]),
    Department(16, "Corporate Communications","comms",     "PR, internal comms, crisis management", ["HuggingFaceFW/fineweb-edu"]),
    Department(17, "Fixed Asset Management",  "assets",    "Asset lifecycle, depreciation, capex", ["infCapital/investopedia_terms_en"]),
    Department(18, "Internal Audit",          "intaudit",  "Internal controls, fraud prevention", ["nvidia/Nemotron-Pretraining-Legal-v1"]),
    Department(19, "Supply Chain Planning",   "scm",       "Demand forecasting, supplier planning", ["HuggingFaceFW/fineweb-edu"]),
    Department(20, "Data Analytics",          "analytics", "Data science, ML models, dashboards", ["transformersbook/codeparrot"]),
    Department(21, "Treasury Management",     "treasury",  "Cash management, liquidity", ["JanosAudran/financial-reports-sec"]),
    Department(22, "Business Intelligence",   "bi",        "Competitive intelligence, market data", ["HuggingFaceFW/fineweb-edu"]),
    Department(23, "Contract Administration",  "contract",  "Contract lifecycle, renewals, compliance", ["nvidia/Nemotron-Pretraining-Legal-v1"]),
    Department(24, "Asset Valuation",          "valuation", "Valuation models, appraisal, impairment", ["JanosAudran/financial-reports-sec"]),
]
NUM_DEPARTMENTS = len(DEPARTMENTS)
DEPT_NAMES = [d.short for d in DEPARTMENTS]

@dataclass
class BlackwellConfig:
    num_gpus: int = 8
    micro_batch_size: int = 2
    grad_accum_steps: int = 8
    global_batch_size: int = 128
    max_seq_len: int = 4096
    adapter_rank: int = 64
    top_k_experts: int = 5
    learning_rate: float = 1e-4
    warmup_steps: int = 500
    weight_decay: float = 0.01
    save_every: int = 100

blackwell = BlackwellConfig()
print(f"{NUM_DEPARTMENTS} departments, top-{blackwell.top_k_experts} routing")

---
## 2. Load Verified Error-Free Datasets

All sources verified working: FineWeb-Edu, OpenWebMath, CodeParrot, Nemotron-Legal, Investopedia, SEC.

In [ ]:
MAX_TOTAL = 300000
train_texts = []

def load_safe(name, ds_fn, key, limit, max_len=2048):
    count = 0
    try:
        ds = ds_fn()
        for x in ds:
            if count >= limit or len(train_texts) >= MAX_TOTAL:
                break
            if key in x and x[key] and isinstance(x[key], str):
                train_texts.append(x[key][:max_len])
                count += 1
    except Exception as e:
        print(f"  WARN: {name} failed — {e}")
    print(f"  {name}: {count} loaded ({len(train_texts)} total)")
    return count

def load_safe_multi(name, ds_fn, fields, limit, max_len=2048):
    count = 0
    try:
        ds = ds_fn()
        for x in ds:
            if count >= limit or len(train_texts) >= MAX_TOTAL:
                break
            for f in fields:
                if f in x and x[f] and isinstance(x[f], str):
                    train_texts.append(x[f][:max_len])
                    count += 1
                    break
    except Exception as e:
        print(f"  WARN: {name} failed — {e}")
    print(f"  {name}: {count} loaded ({len(train_texts)} total)")
    return count

# ─── Verified working datasets ───
print("=== Loading enterprise datasets ===")

# 1. FineWeb-Edu (general enterprise knowledge)
load_safe("FineWeb-Edu", lambda: load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True), "text", 80000)

# 2. OpenWebMath (quantitative reasoning)
load_safe("OpenWebMath", lambda: load_dataset("open-web-math/open-web-math", split="train", streaming=True), "text", 40000)

# 3. CodeParrot (IT/code generation)
load_safe_multi("CodeParrot", lambda: load_dataset("transformersbook/codeparrot", split="train", streaming=True), ["content", "text"], 30000)

# 4. Nemotron-Legal (legal, compliance, audit, contract)
for cfg_name in ["Nemotron-Pretraining-Legal-Case-Law-Summary", "Nemotron-Pretraining-Legal-eCFR", "Nemotron-Pretraining-Legal-CaseHOLD"]:
    load_safe_multi(f"Nemotron-{cfg_name[-20:]}", lambda c=cfg_name: load_dataset("nvidia/Nemotron-Pretraining-Legal-v1", c, split="train", streaming=True), ["text", "input", "content", "question", "answer"], 25000)

# 5. Investopedia (finance, accounting, tax)
load_safe("Investopedia", lambda: load_dataset("infCapital/investopedia_terms_en", split="train", streaming=True), "text", 20000)

# 6. SEC financial reports (finance, audit)
load_safe_multi("SEC-Reports", lambda: load_dataset("JanosAudran/financial-reports-sec", split="train", streaming=True), ["text", "content"], 20000)

# 7. FineWeb (backup general knowledge)
load_safe("FineWeb", lambda: load_dataset("HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True), "text", 30000)

random.seed(42)
random.shuffle(train_texts)
print(f"\n=== TOTAL: {len(train_texts):,} examples ===")

---
## 3. Train Tokenizer

In [ ]:
tok = Tokenizer(models.BPE(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
tok.train_from_iterator(train_texts, trainers.BpeTrainer(vocab_size=16384, special_tokens=["<unk>", "<s>", "</s>", "<pad>"], min_frequency=2))
hf_tokenizer = PreTrainedTokenizerFast(tokenizer_object=tok, unk_token="<unk>", bos_token="<s>", eos_token="</s>", pad_token="<pad>")
print(f"Vocab: {hf_tokenizer.vocab_size}")

---
## 4. Model: Enterprise Multi-Agent System (1 CEO + 24 Depts)

In [ ]:
class TaskRouter(nn.Module):
    def __init__(self, input_dim=4096, hidden_dim=2048, num_experts=24, top_k=5):
        super().__init__()
        self.top_k = top_k
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, num_experts))
    def forward(self, x):
        logits = self.net(x.mean(dim=1))
        vals, idx = torch.topk(logits, self.top_k, dim=-1)
        return F.softmax(vals, dim=-1), idx

class LoRAAdapter(nn.Module):
    def __init__(self, hidden, rank):
        super().__init__()
        self.q = nn.Parameter(torch.zeros(hidden, rank)); self.qb = nn.Parameter(torch.zeros(rank, hidden))
        self.k = nn.Parameter(torch.zeros(hidden, rank)); self.kb = nn.Parameter(torch.zeros(rank, hidden))
        self.v = nn.Parameter(torch.zeros(hidden, rank)); self.vb = nn.Parameter(torch.zeros(rank, hidden))
        self.o = nn.Parameter(torch.zeros(hidden, rank)); self.ob = nn.Parameter(torch.zeros(rank, hidden))
        self.reset()
    def reset(self):
        for p in self.parameters():
            nn.init.kaiming_uniform_(p, a=math.sqrt(5)) if p.dim() >= 2 else nn.init.zeros_(p)
    def forward(self, h, alpha=1.0):
        return ((h @ self.q) @ self.qb + (h @ self.k) @ self.kb + (h @ self.v) @ self.vb + (h @ self.o) @ self.ob) * alpha

class EnterpriseMultiAgentSystem(nn.Module):
    def __init__(self, vocab_size=16384, hidden=4096, num_layers=32, heads=32, rank=64, num_depts=24, top_k=5, max_seq=4096):
        super().__init__()
        self.num_departments = num_depts
        self.top_k = top_k
        self.backbone = LlamaModel(LlamaConfig(vocab_size=vocab_size, hidden_size=hidden, intermediate_size=hidden*4, num_hidden_layers=num_layers, num_attention_heads=heads, max_position_embeddings=max_seq, rope_theta=10000.0, tie_word_embeddings=True, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2"))
        self.lm_head = nn.Linear(hidden, vocab_size, bias=False)
        self.adapters = nn.ModuleDict({d.short: LoRAAdapter(hidden, rank) for d in DEPARTMENTS})
        self.adapters["ceo"] = LoRAAdapter(hidden, rank*2)
        self.router = TaskRouter(input_dim=hidden, num_experts=num_depts, top_k=top_k)
        self.dept_heads = nn.ModuleDict({d.short: nn.Linear(hidden, vocab_size, bias=False) for d in DEPARTMENTS})
        self.ceo_head = nn.Linear(hidden, vocab_size, bias=False)
        for p in self.lm_head.parameters(): nn.init.normal_(p, 0, 0.02)
        for h in self.dept_heads.values():
            nn.init.normal_(h.weight, 0, 0.02)
            if h.bias is not None: nn.init.zeros_(h.bias)
        nn.init.normal_(self.ceo_head.weight, 0, 0.02)

    def forward(self, input_ids, attention_mask=None, task_embeddings=None, active_depts=None):
        h = self.backbone(input_ids, attention_mask=attention_mask).last_hidden_state
        if task_embeddings is None:
            task_embeddings = h[:, 0, :]
        route_w, route_i = self.router(task_embeddings)
        dept_out = {}
        for d in DEPARTMENTS:
            if active_depts and d.short not in active_depts: continue
            dept_out[d.short] = h + self.adapters[d.short](h)
        ceo_h = h + self.adapters["ceo"](h)
        ceo_logits = self.ceo_head(ceo_h)
        dept_logits = {n: self.dept_heads[n](dh) for n, dh in dept_out.items()}
        return ceo_logits, dept_logits, route_w, route_i

    def count_params(self):
        t = sum(p.numel() for p in self.parameters())
        tr = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return t, tr

model = EnterpriseMultiAgentSystem(vocab_size=16384, rank=blackwell.adapter_rank, num_depts=NUM_DEPARTMENTS, top_k=blackwell.top_k_experts).to(torch.bfloat16)
total, trainable = model.count_params()
print(f"Params: {total/1e9:.2f}B total, {trainable/1e9:.2f}B trainable")

---
## 5. Hierarchical Loss

In [ ]:
class EnterpriseHierarchicalLoss(nn.Module):
    def __init__(self, ld=0.3, lc=0.4, lr=0.2, lh=0.1, ignore=-100):
        super().__init__()
        self.ld, self.lc, self.lr, self.lh = ld, lc, lr, lh
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore)
    def forward(self, ceo_logits, dept_logits, route_w, route_i, labels, dept_labels=None, route_targets=None):
        losses = {}
        dl = 0.0
        n = len(dept_logits)
        for dn in dept_logits:
            s = dept_logits[dn][:, :-1, :].contiguous()
            t = (dept_labels or {}).get(dn, labels)[:, 1:].contiguous()
            dl += self.ce(s.view(-1, s.size(-1)), t.view(-1))
        losses["dept"] = dl / max(n, 1)
        s = ceo_logits[:, :-1, :].contiguous()
        t = labels[:, 1:].contiguous()
        losses["ceo"] = self.ce(s.view(-1, s.size(-1)), t.view(-1))
        if route_targets is not None:
            losses["route"] = F.kl_div(F.log_softmax(route_w, dim=-1), F.softmax(route_targets, dim=-1), reduction="batchmean")
        else:
            losses["route"] = torch.tensor(0.0, device=ceo_logits.device)
        losses["total"] = self.ld * losses["dept"] + self.lc * losses["ceo"] + self.lr * losses["route"]
        return losses["total"], losses

criterion = EnterpriseHierarchicalLoss()
print("Loss ready")

---
## 6. Dataset + Tokenize

In [ ]:
from datasets import Dataset as HFDataset

MAX_LENGTH = 512
random.seed(42)
random.shuffle(train_texts)

def encode(texts):
    return hf_tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=False)["input_ids"]

dataset = HFDataset.from_dict({"text": train_texts})
dataset = dataset.map(lambda x: {"input_ids": encode(x["text"])}, remove_columns=["text"], desc="Tokenizing")
dataset = dataset.filter(lambda x: len(x["input_ids"]) > 10, desc="Filtering")
print(f"Examples: {len(dataset)}")

In [ ]:
from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer=hf_tokenizer, mlm=False, pad_to_multiple_of=8)

---
## 7. Training — Save Every 100 Steps, Resume from HF on Failure

- Checkpoints uploaded to HF every 100 steps
- `scheduler.pt` + `rng_state.pth` included for proper resume
- `optimizer.pt` excluded (too large)
- On resume: checks local → checks HF → downloads latest → resumes

In [ ]:
from transformers import TrainingArguments, Trainer, TrainerCallback

MODEL_NAME = "pink-elephant-enterprise-agent"
REPO_ID = f"pinkelephantlimited/{MODEL_NAME}"

api.create_repo(REPO_ID, private=False, repo_type="model", exist_ok=True)
print(f"Repo {REPO_ID} ready")

class HFSaveCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = f"{args.output_dir}/checkpoint-{state.global_step}"
        if os.path.exists(ckpt_dir):
            print(f"\nUploading checkpoint-{state.global_step} to HF...")
            HfApi().upload_folder(folder_path=ckpt_dir, repo_id=REPO_ID, path_in_repo=f"checkpoints/checkpoint-{state.global_step}", ignore_patterns=["*.bin", "optimizer.pt"])
            print(f"  -> Uploaded ({state.global_step} steps)")

args = TrainingArguments(
    output_dir="./" + MODEL_NAME,
    per_device_train_batch_size=blackwell.micro_batch_size,
    gradient_accumulation_steps=blackwell.grad_accum_steps,
    num_train_epochs=10,
    learning_rate=blackwell.learning_rate,
    weight_decay=blackwell.weight_decay,
    warmup_steps=blackwell.warmup_steps,
    logging_steps=10,
    save_strategy="steps",
    save_steps=blackwell.save_every,
    save_total_limit=3,
    report_to="none",
    bf16=True,
    optim="adamw_8bit",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    ddp_find_unused_parameters=False,
)

# ─── Resume: local → HF ───
resume = None
ckpts = sorted(glob.glob(f"./{MODEL_NAME}/checkpoint-*"))
if ckpts:
    resume = ckpts[-1]
    print(f"Resuming from local: {resume}")
else:
    print("No local checkpoints. Checking HF...")
    try:
        files = api.list_repo_files(REPO_ID, repo_type="model")
        ckpt_dirs = set()
        for f in files:
            if f.startswith("checkpoints/checkpoint-"):
                parts = f.split("/")
                if len(parts) >= 2:
                    ckpt_dirs.add(parts[1])
        if ckpt_dirs:
            latest = sorted(ckpt_dirs)[-1]
            dst = f"./{MODEL_NAME}/{latest}"
            os.makedirs(dst, exist_ok=True)
            print(f"Downloading: {latest}")
            for fn in ["config.json", "generation_config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json", "trainer_state.json", "scheduler.pt", "rng_state.pth"]:
                try:
                    p = hf_hub_download(repo_id=REPO_ID, filename=f"checkpoints/{latest}/{fn}", repo_type="model")
                    shutil.copy2(p, os.path.join(dst, fn))
                    print(f"  Got {fn}")
                except Exception as e:
                    print(f"  Skip {fn} ({e})")
            if os.path.exists(os.path.join(dst, "trainer_state.json")):
                resume = dst
                print(f"Resuming from remote: {resume}")
            else:
                print("Download incomplete — starting from scratch")
    except Exception as e:
        print(f"Error: {e}")

# ─── Train ───
trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator, callbacks=[HFSaveCallback])
trainer.train(resume_from_checkpoint=resume)
print("Training complete!")

---
## 8. Upload Final Model

In [ ]:
save_dir = "/tmp/" + MODEL_NAME
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)

model.save_pretrained(save_dir, safe_serialization=True)
hf_tokenizer.save_pretrained(save_dir)

cfg_path = os.path.join(save_dir, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
cfg["_name_or_path"] = f"pinkelephantlimited/{MODEL_NAME}"
cfg["architectures"] = ["LlamaForCausalLM"]
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

api = HfApi()
api.create_repo(REPO_ID, private=False, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=save_dir, repo_id=REPO_ID, ignore_patterns=["*.bin"])
print(f"Uploaded: https://huggingface.co/{REPO_ID}")

---
## 9. Inference Pipeline

In [ ]:
@dataclass
class AgentResponse:
    department: str
    output: str
    confidence: float
    routing_weight: float

@dataclass
class EnterpriseDecision:
    objective: str
    ceo_plan: str
    department_responses: List[AgentResponse]
    ceo_decision: str
    confidence: float
    refinement_rounds: int

class EnterpriseInferencePipeline:
    def __init__(self, model, tokenizer, max_refine=3, conf_thresh=0.85):
        self.model = model
        self.tokenizer = tokenizer
        self.max_refine = max_refine
        self.conf_thresh = conf_thresh

    def __call__(self, objective, verbose=True):
        ceo_plan = self._gen(f"<|ceo|> Decompose: {objective}\nPlan:<|end|>")
        if verbose:
            print(f"\n{'='*60}\nCEO PLAN: {objective}\n{'='*60}\n{ceo_plan}\n")
        depts, weights = self._route(objective)
        if verbose:
            print(f"Routing: {len(depts)} depts activated")
            for d, w in zip(depts, weights):
                print(f"  {d.name}: {w:.2%}")
        responses = []
        for d, w in zip(depts, weights):
            out = self._gen(f"<|{d.short}|> As {d.name}, analyze:\nObjective: {objective}\nContext: {ceo_plan}\nAnalysis:<|end|>")
            responses.append(AgentResponse(d.short, out, 0.5, w))
            if verbose:
                print(f"\n  [{d.name}] {out[:200]}...")
        decision = self._gen(f"<|decision|> Synthesize:\n{chr(10).join([f'[{r.department}]: {r.output[:200]}' for r in responses])}\nFinal decision:<|end|>")
        conf = sum(r.confidence for r in responses) / len(responses) * 0.7 + 0.3
        if verbose:
            print(f"\n{'='*60}\nDECISION (conf: {conf:.1%})\n{'='*60}\n{decision}\n")
        rounds = 0
        while conf < self.conf_thresh and rounds < self.max_refine:
            rounds += 1
            for r in responses:
                if r.confidence < 0.5:
                    r.output = self._gen(f"<|{r.department}|> Refine:\nObjective: {objective}\nCurrent: {decision}\nRefined:<|end|>")
            decision = self._gen(f"<|decision|> Final after refinement:\n{chr(10).join([f'[{r.department}]: {r.output[:200]}' for r in responses])}\nFinal:<|end|>")
            conf = sum(r.confidence for r in responses) / len(responses) * 0.7 + 0.3
        return EnterpriseDecision(objective, ceo_plan, responses, decision, conf, rounds)

    def _route(self, objective):
        tokens = self.tokenizer.encode(objective, return_tensors="pt").cuda()
        with torch.no_grad(), autocast(dtype=torch.bfloat16):
            h = self.model.backbone(tokens).last_hidden_state
            w, i = self.model.router(h[:, 0, :])
        return [DEPARTMENTS[idx] for idx in i[0].tolist()], w[0].tolist()

    def _gen(self, prompt, max_tokens=256):
        tokens = self.tokenizer.encode(prompt, return_tensors="pt").cuda()
        with torch.no_grad(), autocast(dtype=torch.bfloat16):
            out = self.model.backbone.generate(tokens, max_new_tokens=max_tokens, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=self.tokenizer.pad_token_id)
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

print("Inference pipeline defined")

# Demo (requires trained model):
# pipeline = EnterpriseInferencePipeline(model, hf_tokenizer)
# result = pipeline("Launch a new product line in Southeast Asia with $5M budget")
# print(result.ceo_decision)

In [ ]:
print("="*60)
print("Enterprise Multi-Agent AI System — Blackwell 6000")
print("="*60)
print(f"{NUM_DEPARTMENTS} departments, 1 CEO, top-{blackwell.top_k_experts} routing")
print(f"Params: {total/1e9:.2f}B total")
print(f"Checkpoints: every {blackwell.save_every} steps → HF")